# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

Runtime → Change runtime type → **T4 GPU** → Save. Phần này: lấy repo, cài thư viện, nạp Kaggle Secrets,
mount Drive (lưu adapter/checkpoint), rồi chạy `pytest` để chắc công cụ của đề chạy đúng.

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L

# Mở notebook từ GitHub thì Colab đứng ở /content, chưa có scripts/ → clone repo và cd vào.
import os
REPO_URL = "https://github.com/SxAinsworth/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if not os.path.exists("scripts/score.py"):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
print("cwd:", os.getcwd())

In [ ]:
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print("Kaggle user:", os.environ['KAGGLE_USERNAME'])

In [ ]:
# Mount Google Drive để sao lưu adapter LoRA / projector / predictions sau mỗi lần chạy
# (session Colab miễn phí có thể bị ngắt giữa chừng).
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/lab_day10")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Sao lưu vào:", DRIVE_DIR)

In [ ]:
# Kiểm tra công cụ của đề. Kỳ vọng: 68 passed, 1 skipped (test dữ liệu thật bị bỏ qua khi chưa tải dataset).
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải dataset Kaggle và chia split theo phiên quay (cố định, ai chạy cũng ra cùng kết quả).
# Bỏ qua nếu đã có data/ (chạy lại notebook sau khi session bị ngắt mà runtime còn giữ ổ đĩa).
# Không có Kaggle? Thay bằng: !python scripts/prepare_data.py --zip <đường_dẫn_zip>
import os
if not os.path.exists("data/splits/meta.json"):
    !python scripts/prepare_data.py --kaggle
else:
    print("Đã có data/splits/meta.json, bỏ qua bước tải.")

In [ ]:
# Thống kê từng split + kiểm tra rò rỉ phiên quay (số liệu cho câu 6 của REPORT).
import json
from collections import Counter
from scripts.labcommon import CLASSES, read_jsonl

SPLITS = ("train", "val", "test")
data = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in SPLITS}
meta = json.load(open("data/splits/meta.json", encoding="utf-8"))
DATASET_VERSION = meta["dataset_version"]
print("dataset_version:", DATASET_VERSION)
assert DATASET_VERSION.startswith("d10-"), DATASET_VERSION

# Bảng chuẩn của đề (README mục 3): ảnh, pothole, crack, manhole
EXPECTED = {"train": (1422, 1074, 1750, 691), "val": (276, 66, 470, 103), "test": (311, 121, 299, 163)}
print(f"{'split':6s} {'ảnh':>5s} {'sessions':>8s} " + " ".join(f"{c:>8s}" for c in CLASSES) + f" {'box/ảnh':>8s} {'ảnh rỗng':>8s}")
for s in SPLITS:
    rows = data[s]
    box = Counter(o["label"] for r in rows for o in r["objects"])
    n_box = sum(box.values())
    n_empty = sum(1 for r in rows if not r["objects"])
    n_sess = len({r["session"] for r in rows})
    print(f"{s:6s} {len(rows):5d} {n_sess:8d} " + " ".join(f"{box[c]:8d}" for c in CLASSES)
          + f" {n_box / len(rows):8.2f} {n_empty:8d}")
    got = (len(rows), *(box[c] for c in CLASSES))
    assert got == EXPECTED[s], f"{s}: {got} != {EXPECTED[s]}"

# Không session nào nằm ở 2 split
sess = {s: {r["session"] for r in data[s]} for s in SPLITS}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    inter = sess[a] & sess[b]
    assert not inter, f"Rò rỉ phiên quay {a}/{b}: {sorted(inter)[:5]}"
print("OK: số ảnh/box khớp bảng của đề; không session nào xuất hiện ở hai split.")

# Kích thước box: diện tích tương đối (w*h) theo lớp, trên train (dùng cho câu 9)
import statistics
for c in CLASSES:
    areas = [o["yolo"][2] * o["yolo"][3] for r in data["train"] for o in r["objects"] if o["label"] == c]
    print(f"  {c:8s} diện tích trung vị = {100 * statistics.median(areas):.2f}% khung hình")

In [ ]:
# Minh hoạ vì sao KHÔNG chia ngẫu nhiên: nếu chia theo ảnh (cùng tỉ lệ ~15% test),
# bao nhiêu ảnh test có "anh em" cùng phiên quay nằm trong train? (số liệu cho câu 6)
import random
all_rows = [r for s in SPLITS for r in data[s]]
rng = random.Random(0)
shuffled = all_rows[:]
rng.shuffle(shuffled)
n_test = len(data["test"])
rand_test, rand_train = shuffled[:n_test], shuffled[n_test:]
train_sess = {r["session"] for r in rand_train}
leaked = sum(1 for r in rand_test if r["session"] in train_sess)
print(f"Chia ngẫu nhiên: {leaked}/{n_test} ảnh test ({100 * leaked / n_test:.1f}%) có cùng phiên quay với ảnh train.")
print(f"Chia theo phiên (prepare_data.py): 0/{n_test} ảnh test.")
print("Số phiên quay:", len({r['session'] for r in all_rows}), "| ảnh/phiên trung bình:",
      round(len(all_rows) / len({r['session'] for r in all_rows}), 1))

Vẽ 6 ảnh train kèm box gốc (YOLO → pixel) để chắc hiểu nhãn. Ảnh được lưu vào `report/figures/data_samples.png`.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def draw_yolo(img, objects, width=3):
    """Vẽ box YOLO (xc, yc, w, h chuẩn hoá [0,1]) lên bản sao của ảnh."""
    img = img.copy()
    d = ImageDraw.Draw(img)
    W, H = img.size
    for o in objects:
        xc, yc, w, h = o["yolo"]
        box = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
        d.rectangle(box, outline=COLORS[o["label"]], width=width)
        d.text((box[0] + 2, box[1] + 2), o["label"], fill=COLORS[o["label"]])
    return img

rng = random.Random(10)
# Lấy mẫu có đủ cả 3 lớp: 2 ảnh có pothole, 2 có crack, 2 có manhole
samples = []
for c in CLASSES:
    pool = [r for r in data["train"] if any(o["label"] == c for o in r["objects"]) and r not in samples]
    samples += rng.sample(pool, 2)

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, samples):
    img = Image.open(Path("data") / r["image_path"]).convert("RGB")
    ax.imshow(draw_yolo(img, r["objects"]))
    ax.set_title(f"{r['image_id']}\n{r['session']} · {len(r['objects'])} box", fontsize=9)
    ax.axis("off")
fig.suptitle("Nhãn gốc (đỏ=pothole, vàng=crack, xanh=manhole)")
plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("report/figures/data_samples.png", dpi=80, bbox_inches="tight")
plt.show()
print("Kích thước ảnh (W, H):", Counter((r["width"], r["height"]) for r in all_rows))

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9